# 04 — End-to-end clinical scenarios

Phase 11.6 demonstrates **five end-to-end multi-tool clinical scenarios** running on the same federation that powers the production agent.

Scenarios:
1. **acute_stroke_lvo** — acute stroke + LVO + reperfusion eligibility
2. **sepsis_bundle** — sepsis bundle + antibiogram-driven empiric pick
3. **polytrauma_mtp** — polytrauma + MTP activation
4. **geriatric_polypharmacy** — geriatric polypharmacy med review
5. **mental_health_crisis** — mental-health crisis + admission decision

Each scenario chains **5-7 deterministic-floor MCP tools**. No FHIR network calls — inputs inline so the runs are reproducible in CI.

In [ ]:
import asyncio
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from a2a_agent.clinical_scenarios import (
    list_scenarios, run_all_scenarios, run_scenario,
)

list_scenarios()

## Run a single scenario

`run_scenario` returns a `ScenarioRun` with the ordered list of `ScenarioStep`s, each carrying the tool name, a one-line summary, and the structured Pydantic output.

In [ ]:
stroke = asyncio.run(run_scenario('acute_stroke_lvo'))
print(f"{stroke.title}")
print(f"Tools invoked: {stroke.n_tools_invoked}")
print()
for i, step in enumerate(stroke.steps, 1):
    print(f"{i}. {step.tool}")
    print(f"   {step.summary}")
print()
print(f"FINAL: {stroke.final_summary}")

## Run all 5 scenarios

In [ ]:
runs = asyncio.run(run_all_scenarios())
for r in runs:
    print(f"--- {r.scenario_id} ({r.n_tools_invoked} tools) ---")
    print(r.final_summary)
    print()
total_calls = sum(r.n_tools_invoked for r in runs)
print(f"Cumulative tool calls across all 5 scenarios: {total_calls}")

## Inspect a per-scenario step output

Every `ScenarioStep.output` is a Pydantic model — it can be serialised, compared structurally, or fed forward into another tool.

In [ ]:
geri = asyncio.run(run_scenario('geriatric_polypharmacy'))
poly_step = next(s for s in geri.steps if s.tool == 'detect_polypharmacy_concerns')
print(f"Polypharmacy severity: {poly_step.output.polypharmacy_severity}")
print(f"Number of interactions: {len(poly_step.output.interactions)}")
for ddi in poly_step.output.interactions[:3]:
    print(f"  - {ddi.drug_a} + {ddi.drug_b} ({ddi.severity})")

## Notes

- The smoke test `tests/integration/test_clinical_scenarios.py` runs all five scenarios on every CI build.
- Inputs are intentionally inline; production deployments would replace the patient-state arguments with FHIR Bundle extracts (`fetch_patient_bundle` from the SHARP context).
- See `docs/scenarios/` for the per-scenario design rationale + clinical references.